# Technical Assessment — Credit Default Prediction
### PrivatBank · Binary Classification · Imbalanced Learning

---

## 📋 Executive Summary

> **Goal:** Build a binary classifier to predict credit default (`gb = 1`) on heavily imbalanced data with a grouped client structure.

| | Logistic Regression (baseline) | CatBoost + Optuna (final) |
|---|---|---|
| **PR-AUC (OOF)** | ~0.030 | **~0.199** |
| **ROC-AUC (OOF)** | ~0.540 | **~0.821** |
| **Improvement** | — | **~6.6× in PR-AUC** |

**Key engineering decisions:**

| Challenge | Solution |
|---|---|
| Client appears in multiple rows → data leakage | `StratifiedGroupKFold` by `id` |
| ~9% default rate (severe imbalance) | PR-AUC as primary metric + `scale_pos_weight` |
| High-cardinality categorical features | Native CatBoost CTR encoding |
| Overfitting risk during HPO | `MedianPruner` + fold-0 warmup constraint |
| Skewed numerics breaking LR | `RobustScaler` (IQR-based) |

---

## 1. Imports & Configuration

In [ ]:
!pip install numpy pandas matplotlib seaborn scipy scikit-learn catboost optuna

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import re
import warnings
from typing import List, Tuple

# ── Third-party: data & viz ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ── Third-party: statistics ───────────────────────────────────────────────────
from scipy.stats import chi2_contingency, mannwhitneyu

# ── Third-party: ML ──────────────────────────────────────────────────────────
import optuna
from catboost import CatBoostClassifier, Pool
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")

# ── Global constants ──────────────────────────────────────────────────────────
DATA_PATH    = "train_df.csv"
TARGET_COL   = "gb"
ID_COL       = "id"
RANDOM_STATE = 42
N_SPLITS     = 5

# ── Colour palette (used consistently across all plots) ───────────────────────
PALETTE = {
    "primary":   "#2E86AB",
    "secondary": "#A23B72",
    "accent":    "#F18F01",
    "pos":       "#C73E1D",
    "neg":       "#3A7D44",
}

# ── Matplotlib / Seaborn defaults ─────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette([PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]])
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 13, "axes.labelsize": 11})

# ── Optuna flag & pre-cached best parameters ──────────────────────────────────
# Set RUN_OPTUNA = True to re-run the 50-trial search (takes ~20–40 min).
# When False the pipeline falls back to DEFAULT_BEST_PARAMS automatically.
RUN_OPTUNA = False

DEFAULT_BEST_PARAMS = {
    "depth":             6,
    "learning_rate":     0.018669,
    "l2_leaf_reg":       11.6927,
    "random_strength":   0.6015,
    "min_data_in_leaf":  34,
    "max_ctr_complexity": 2,
    "one_hot_max_size":  7,
    "model_size_reg":    4.9675,
    "scale_pos_weight":  10.0958,
    "loss_function":     "Logloss",
}

np.random.seed(RANDOM_STATE)

## 2. Data Loading

In [ ]:
df = pd.read_csv("train_df.csv", sep="\t")

num_cols = sorted(c for c in df.columns if c.startswith("num_"))
cat_cols = sorted(c for c in df.columns if c.startswith("cat_"))
feature_cols = num_cols + cat_cols

# Basic data quality assertions
assert df.shape[0] > 0, "Loaded dataframe is empty!"
assert TARGET_COL in df.columns, f"Target column '{TARGET_COL}' not found!"

print(f"Dataset shape   : {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"Unique clients  : {df[ID_COL].nunique():,}")
print(f"Numerical feats : {len(num_cols)}")
print(f"Categorical feats: {len(cat_cols)}")
print()
print(df[feature_cols].dtypes.value_counts().to_string())

## 3. Exploratory Data Analysis (EDA)

The dataset has a **grouped structure**: one client (`id`) may appear
across multiple rows (e.g. different loan applications over time).
A naive `train_test_split` would place the same client in both train
and validation, causing **data leakage**.  
We must use **group-aware splitting** (`StratifiedGroupKFold` by `id`).

### 3.1 Class Imbalance

In [ ]:
target_counts   = df[TARGET_COL].value_counts().sort_index()
pos_rate        = df[TARGET_COL].mean()
imbalance_ratio = target_counts[0] / target_counts[1]

print(f"Positive rate (default=1) : {pos_rate:.2%}")
print(f"Imbalance ratio  (0:1)    : {imbalance_ratio:.1f}:1")

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    ["Class 0 — Non-Default", "Class 1 — Default"],
    target_counts.values,
    color=[PALETTE["neg"], PALETTE["pos"]],
    edgecolor="white",
    width=0.5,
)
ax.bar_label(bars, padding=4, fmt="%d")
ax.set_title("Class Distribution", fontsize=12, fontweight="bold")
ax.set_xlabel("Target Class (0 = Non-Default, 1 = Default)", fontsize=11)
ax.set_ylabel("Number of Records", fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

**Observation:** The dataset is severely imbalanced (~9% defaults).
This rules out Accuracy as a metric and motivates the following choices:

- **Primary metric — PR-AUC**: focuses on the minority (Default) class,
  unaffected by the large volume of true negatives.
- **Secondary — F1 / Precision / Recall**: evaluated at the optimal
  decision threshold to allow flexible business tuning (shift threshold
  to maximise Recall when cost of missed default > cost of false alarm).

### 3.2 Missing Values

In [ ]:
missing_pct = df[feature_cols].isnull().mean() * 100
empty_cols  = missing_pct[missing_pct >= 100].index.tolist()

print(f"Features with 100% missing values: {len(empty_cols)}")
print(f"Features with any missing values : {(missing_pct > 0).sum()}")

Fully-empty columns carry **zero variance** and zero predictive power.
Removing them reduces memory footprint and prevents the model from
learning noise on null tokens.

### 3.3 Numerical Feature Skewness

In [ ]:
# Define once — reused in the Mann-Whitney test below
valid_num_cols = [c for c in num_cols if df[c].notna().sum() > 100]

skew_series = df[valid_num_cols].skew().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(skew_series.clip(-20, 20), bins=40,
        color=PALETTE["primary"], edgecolor="white")
ax.axvline(0, color="darkred", linestyle="--", linewidth=1.2, label="Perfect symmetry")
ax.set_title("Numerical Feature Skewness Distribution", fontsize=11, fontweight="bold")
ax.set_xlabel("Skewness value")
ax.set_ylabel("Feature count")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

**Implication for model choice:**

| Model family | Sensitivity to skewness | Mitigation |
|---|---|---|
| CatBoost / tree-based | **None** — splits on thresholds only | — |
| Logistic Regression | **High** — raw values multiplied by weights | `RobustScaler` (median + IQR) |

`RobustScaler` is preferred over `StandardScaler` because it uses the
**interquartile range**, making it resistant to extreme outliers that
would otherwise distort the linear model's weight surface.

### 3.4 Categorical Feature Cardinality

In [ ]:
cardinality = df[cat_cols].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(cardinality.clip(0, 100), bins=50,
        color=PALETTE["secondary"], edgecolor="white")
ax.set_xlabel("Number of unique values")
ax.set_ylabel("Number of features")
ax.set_title("Categorical Feature Cardinality (clipped at 100)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

print(f"High-cardinality features (>50 unique values): "
      f"{(cardinality > 50).sum()}")

High-cardinality categoricals are a **risk factor** for One-Hot
Encoding (memory explosion, overfitting). CatBoost's native CTR
(Category Target Rate) encoding handles them gracefully without
manual binning.

### 3.5 Univariate Statistical Tests — Numerical Features (Mann-Whitney U)

In [ ]:
mw_results = []

for col in valid_num_cols:
    vals_0 = df.loc[df[TARGET_COL] == 0, col].dropna()
    vals_1 = df.loc[df[TARGET_COL] == 1, col].dropna()

    if len(vals_0) < 10 or len(vals_1) < 10:
        continue

    stat, pval = mannwhitneyu(vals_0, vals_1, alternative="two-sided")
    mw_results.append({
        "column":    col,
        "p_value":   pval,
        "median_gb0": vals_0.median(),
        "median_gb1": vals_1.median(),
    })

mw_df      = pd.DataFrame(mw_results).sort_values("p_value")
significant = mw_df[mw_df["p_value"] < 0.05]

print(f"[Mann-Whitney U] Features evaluated        : {len(mw_df)}")
print(f"[Mann-Whitney U] Significant (p < 0.05)    : {len(significant)}")
display(significant.head(15))

The **Mann-Whitney U test** is a non-parametric rank-based test —
no normality assumption required. Features with $p < 0.05$ exhibit a
statistically significant **median shift** between Non-Default and
Default classes and are prioritised for the Logistic Regression feature
space (which is sensitive to input dimensionality).

### 3.6 Univariate Statistical Tests — Categorical Features (Chi-squared)

In [ ]:
chi2_results = []
for col in cat_cols:
    ct = pd.crosstab(df[col], df[TARGET_COL])
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        continue
    chi2, pval, dof, _ = chi2_contingency(ct)
    chi2_results.append({
        "column":      col,
        "chi2":        chi2,
        "p_value":     pval,
        "dof":         dof,
        "cardinality": df[col].nunique(),
    })

chi2_df  = pd.DataFrame(chi2_results).sort_values("p_value")
chi2_sig = chi2_df[chi2_df["p_value"] < 0.05]

print(f"[Chi-squared] Categorical features evaluated  : {len(chi2_df)}")
print(f"[Chi-squared] Significant (p < 0.05)          : {len(chi2_sig)}")
display(chi2_df.head(15))

### 📊 EDA Summary

| Insight | Value |
|---|---|
| Severe class imbalance | ~9% defaults, ratio ~10:1 |
| Fully empty features | Identified and scheduled for removal |
| Significant numerical features (MW, p<0.05) | To be confirmed by cell output |
| Significant categorical features (χ², p<0.05) | To be confirmed by cell output |
| High skewness in numerics | Requires `RobustScaler` for LR |
| High-cardinality categoricals | Handled natively by CatBoost CTR |

**Next step:** Feature pruning (empty + duplicate columns) and
pipeline-specific preprocessing per model family.

---

## 4. Data Preprocessing & Cleaning

In [ ]:
# ── Step 1: Drop fully empty columns ─────────────────────────────────────────
cols_before = len(df.columns)
df = df.drop(columns=empty_cols, errors="ignore")
print(f"Dropped {cols_before - len(df.columns)} columns with 100% missing values.")

# ── Step 2: Refresh column lists after drop ───────────────────────────────────
cat_cols = sorted(c for c in df.columns if c.startswith("cat_"))
num_cols = sorted(c for c in df.columns if c.startswith("num_"))
feature_cols = cat_cols + num_cols

# ── Step 3: Detect and remove exact duplicate columns ─────────────────────────
dup_mask = df[feature_cols].T.duplicated(keep="first")
dup_cols = dup_mask[dup_mask].index.tolist()
df = df.drop(columns=dup_cols, errors="ignore")
print(f"Dropped {len(dup_cols)} exact duplicate columns.")

# ── Step 4: Final column registry ────────────────────────────────────────────
cat_cols      = sorted(c for c in df.columns if c.startswith("cat_"))
num_cols      = sorted(c for c in df.columns if c.startswith("num_"))
FINAL_FEATURES = cat_cols + num_cols

print(f"\nFinal feature count: {len(FINAL_FEATURES)} "
      f"({len(num_cols)} numerical, {len(cat_cols)} categorical)")

**Pruning rationale:**

- **Empty columns:** zero variance → zero predictive power, removed to
  reduce memory and prevent noise learning.
- **Exact duplicates:** detected by transposing the feature matrix and
  checking row-level equality. Duplicates cause multicollinearity in
  linear models and inflate tree search space without adding signal.

In [ ]:
# ── Encode categorical NaN as -1 (CatBoost-safe sentinel) ────────────────────
for col in cat_cols:
    df[col] = df[col].fillna(-1).astype(int)

# ── Build global arrays ───────────────────────────────────────────────────────
X_all      = df[FINAL_FEATURES].copy()
y_all      = df[TARGET_COL].values
groups_all = df[ID_COL].values

final_num  = [c for c in FINAL_FEATURES if c.startswith("num_")]
final_cat  = [c for c in FINAL_FEATURES if c.startswith("cat_")]
final_mnar = [c for c in final_num if X_all[c].isnull().any()]

print("Global Preprocessing Summary:")
print(f"  Dataset shape              : {X_all.shape}")
print(f"  Numerical features         : {len(final_num)}")
print(f"  Categorical features        : {len(final_cat)}")
print(f"  MNAR numerical columns      : {len(final_mnar)}")
print(f"  Target class distribution   : "
      f"{pd.Series(y_all).value_counts(normalize=True).round(4).to_dict()}")

Missing categorical values are encoded as `-1` — a reserved sentinel
token. For CatBoost this is treated as a distinct valid category.
For Logistic Regression, `-1` is handled downstream via explicit
label-encoding fitted on training data only (no leakage).

MNAR (Missing Not At Random) numerical columns receive explicit
missingness indicator features in the LR pipeline, preserving the
information carried by the absence pattern itself.

---

## 5. Helper Functions & Preprocessing Pipelines

All utility functions are centralised here to keep the modelling loops
clean and auditable. Each function is stateless and leakage-safe
(all transformations are fit on training data only).

In [ ]:
def sanitize_names(cols: List[str]) -> List[str]:
    """Remove special characters from column names for CatBoost compatibility."""
    return [re.sub(r"[^\w]", "_", c) for c in cols]


def encode_categoricals(
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    cat_cols: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Leakage-safe label encoding: categories fitted on training fold only.
    Unseen validation tokens are mapped to -1.
    """
    X_tr, X_va = X_train.copy(), X_val.copy()
    for col in cat_cols:
        if col not in X_tr.columns:
            continue
        uniques = X_tr[col].astype(str).fillna("__NA__").unique()
        mapping = {v: i for i, v in enumerate(sorted(uniques))}
        X_tr[col] = X_tr[col].astype(str).fillna("__NA__").map(mapping).astype(int)
        X_va[col] = (
            X_va[col].astype(str).fillna("__NA__").map(mapping).fillna(-1).astype(int)
        )
    return X_tr, X_va


def prepare_for_logreg(
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    mnar_cols: List[str],
    num_cols: List[str],
    cat_cols: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Full preprocessing pipeline for Logistic Regression:
    1. MNAR missingness indicators
    2. Leakage-safe categorical encoding
    3. Median imputation (train medians only)
    4. RobustScaler (IQR-based, outlier-resistant)
    """
    X_tr, X_va = X_train.copy(), X_val.copy()

    # 1. MNAR indicators
    for col in mnar_cols:
        if col in X_tr.columns:
            X_tr[f"{col}__miss"] = X_tr[col].isnull().astype(np.int8)
            X_va[f"{col}__miss"] = X_va[col].isnull().astype(np.int8)

    # 2. Categorical encoding
    X_tr, X_va = encode_categoricals(X_tr, X_va, cat_cols)

    # 3. Numerical imputation via training medians
    medians = X_tr[num_cols].median().fillna(0)
    X_tr[num_cols] = X_tr[num_cols].fillna(medians)
    X_va[num_cols] = X_va[num_cols].fillna(medians)

    # 4. Robust scaling
    all_cols = X_tr.columns.tolist()
    scaler   = RobustScaler()
    X_tr_s   = pd.DataFrame(
        scaler.fit_transform(X_tr), columns=all_cols, index=X_tr.index
    )
    X_va_s   = pd.DataFrame(
        scaler.transform(X_va), columns=all_cols, index=X_val.index
    )
    return X_tr_s, X_va_s


def prepare_for_trees(
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    cat_cols: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Minimal preprocessing for tree-based models: leakage-safe label encoding.
    Numerics are passed as-is (trees are scale- and skew-invariant).
    """
    return encode_categoricals(X_train, X_val, cat_cols)


def find_optimal_f1(
    y_true: np.ndarray,
    y_prob: np.ndarray,
) -> Tuple[float, float]:
    """Vectorised search for the decision threshold maximising F1.
    Uses the full precision-recall curve instead of a manual linspace loop.
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-10)
    best_idx  = np.argmax(f1_scores)
    best_t    = thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1]
    return float(best_t), float(f1_scores[best_idx])

print("Helper functions registered successfully.")

## 6. Cross-Validation Strategy

In [ ]:
sgkf   = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
splits = list(sgkf.split(X_all, y_all, groups=groups_all))

print("StratifiedGroupKFold — Fold Summary:")
print(f"{'Fold':<6} {'Train rows':>12} {'Train entities':>16} "
      f"{'Val rows':>10} {'Val entities':>14} {'Val default rate':>18}")
print("-" * 80)

for fold_idx, (train_idx, val_idx) in enumerate(splits):
    train_entities = len(np.unique(groups_all[train_idx]))
    val_entities   = len(np.unique(groups_all[val_idx]))
    val_default    = y_all[val_idx].mean()
    print(
        f"{fold_idx:<6} {len(train_idx):>12,} {train_entities:>16,} "
        f"{len(val_idx):>10,} {val_entities:>14,} {val_default:>17.2%}"
    )

**Why `StratifiedGroupKFold`?**

`StratifiedGroupKFold` enforces two constraints simultaneously:

1. **Group isolation** — all rows belonging to one client (`id`) land in
   the same fold. This guarantees **zero identity overlap** between train
   and validation, which standard K-Fold cannot ensure.
2. **Class stratification** — each fold preserves the global default rate,
   preventing lucky/unlucky splits that would artificially inflate or
   deflate metric estimates.

---

## 7. Baseline Model: Logistic Regression

We start with a **linear baseline** to establish a lower-bound benchmark.
Features are filtered to the statistically significant subset identified
in EDA (Mann-Whitney for numerics, Chi-squared for categoricals),
reducing dimensionality and improving convergence.

In [ ]:
# ── Statistically significant feature subsets ─────────────────────────────────
sig_num  = [c for c in significant["column"].tolist() if c in X_all.columns]
sig_cat  = [c for c in chi2_sig["column"].tolist()   if c in X_all.columns]
sig_mnar = [c for c in sig_num if X_all[c].isnull().any()]

print(f"Numerical features retained  : {len(sig_num):>4}  (from {len(final_num)})")
print(f"Categorical features retained: {len(sig_cat):>4}  (from {len(final_cat)})")
print(f"MNAR risk indicators (num)   : {len(sig_mnar):>4}")

sig_features = sig_num + sig_cat
X_all_lr     = X_all[sig_features].copy()

In [ ]:
# ── Cross-validation loop ─────────────────────────────────────────────────────
oof_lr         = np.full(len(y_all), np.nan)
lr_fold_metrics = []
lr_curve_data   = {"folds": {}}

for fold_idx, (train_idx, val_idx) in enumerate(splits):
    X_tr_raw, X_va_raw = X_all_lr.iloc[train_idx], X_all_lr.iloc[val_idx]
    y_tr, y_va         = y_all[train_idx], y_all[val_idx]

    X_tr_lr, X_va_lr = prepare_for_logreg(
        X_tr_raw, X_va_raw, sig_mnar, sig_num, sig_cat
    )

    lr = LogisticRegression(
        C=0.1,
        penalty="l2",
        solver="saga",
        max_iter=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    lr.fit(X_tr_lr, y_tr)
    val_prob = lr.predict_proba(X_va_lr)[:, 1]
    oof_lr[val_idx] = val_prob

    pr_auc    = average_precision_score(y_va, val_prob)
    roc_auc   = roc_auc_score(y_va, val_prob)
    best_t, best_f1 = find_optimal_f1(y_va, val_prob)
    y_pred    = (val_prob >= best_t).astype(int)
    precision = precision_score(y_va, y_pred, zero_division=0)
    recall    = recall_score(y_va, y_pred, zero_division=0)

    lr_fold_metrics.append({
        "fold": fold_idx, "pr_auc": pr_auc, "roc_auc": roc_auc,
        "precision": precision, "recall": recall,
        "f1": best_f1, "threshold": best_t,
    })

    prec_c, rec_c, _ = precision_recall_curve(y_va, val_prob)
    fpr, tpr, _      = roc_curve(y_va, val_prob)
    lr_curve_data["folds"][fold_idx] = {
        "prec": prec_c, "rec": rec_c, "fpr": fpr, "tpr": tpr,
        "pr_auc": pr_auc, "roc_auc": roc_auc,
    }
    print(
        f"  Fold {fold_idx}: PR-AUC={pr_auc:.4f} | ROC-AUC={roc_auc:.4f} | "
        f"F1={best_f1:.4f} [threshold={best_t:.3f}]"
    )

lr_metrics_df = pd.DataFrame(lr_fold_metrics)

In [ ]:
# ── PR Curves across folds ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
baseline_rate = y_all.mean()

for fold_idx, data in lr_curve_data["folds"].items():
    ax.plot(data["rec"], data["prec"], alpha=0.7, linewidth=2,
            label=f"Fold {fold_idx} (AUC={data['pr_auc']:.3f})")

ax.axhline(baseline_rate, color="darkred", linestyle="--", linewidth=1.5,
           label=f"Baseline rate ({baseline_rate:.3%})")
ax.set_title("Logistic Regression — Precision-Recall Curves (per fold)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Recall (True Positive Rate)")
ax.set_ylabel("Precision (Positive Predictive Value)")
ax.legend(frameon=False, fontsize=9, loc="upper right")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.0])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="both", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curves across folds ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

for fold_idx, data in lr_curve_data["folds"].items():
    ax.plot(data["fpr"], data["tpr"], alpha=0.7, linewidth=2,
            label=f"Fold {fold_idx} (AUC={data['roc_auc']:.3f})")

ax.plot([0, 1], [0, 1], color="gray", linestyle="--",
        linewidth=1.5, label="Random guess baseline")
ax.set_title("Logistic Regression — ROC Curves (per fold)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("False Positive Rate (FPR)")
ax.set_ylabel("True Positive Rate (TPR / Recall)")
ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.0])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="both", alpha=0.2)
plt.tight_layout()
plt.show()

### 7. Baseline Conclusion

The Logistic Regression baseline demonstrates **structurally poor
predictive power** across all folds:

- **PR-AUC** hovers in the `0.016–0.047` range — barely above the
  naive baseline rate, indicating the model cannot meaningfully separate
  default from non-default distributions.
- **ROC-AUC** of `0.51–0.62` confirms discriminative capacity only
  marginally above random guessing.

**Root cause:** The data contains highly non-linear interactions between
features that a linear decision boundary cannot capture — regardless
of regularisation or class weighting.  
**Next step:** Non-linear gradient-boosted trees (CatBoost).

---

## 8. CatBoost + Optuna Hyperparameter Tuning

### 8.1 Pre-compute CatBoost Pools

To maximise training efficiency during the Optuna search, we
**pre-compile and cache** each fold's data into native CatBoost
`Pool` binary structures. This step runs once — preventing redundant
data transformations across 50+ trials and cutting total pipeline
latency significantly.

In [ ]:
cat_feature_indices = [FINAL_FEATURES.index(c) for c in final_cat if c in FINAL_FEATURES]
precomputed_pools   = []

print("Pre-compiling CatBoost Pools across validation splits...")

for fold_idx, (train_idx, val_idx) in enumerate(splits):
    X_tr_raw, X_va_raw = X_all.iloc[train_idx], X_all.iloc[val_idx]
    y_tr, y_va         = y_all[train_idx], y_all[val_idx]

    X_tr_cb, X_va_cb = prepare_for_trees(X_tr_raw, X_va_raw, final_cat)

    safe_names = sanitize_names(X_tr_cb.columns.tolist())
    X_tr_cb.columns = safe_names
    X_va_cb.columns = safe_names

    train_pool = Pool(X_tr_cb, y_tr, cat_features=cat_feature_indices)
    val_pool   = Pool(X_va_cb, y_va, cat_features=cat_feature_indices)
    precomputed_pools.append((train_pool, val_pool, y_va))
    print(f"  Fold {fold_idx}: Pool cached successfully.")

### 8.2 Optuna Objective Function

The objective searches over tree architecture (depth, learning rate,
L2 regularisation) and two imbalance-handling strategies:
`auto_class_weights` (Balanced/SqrtBalanced) or manual `scale_pos_weight`.
**Target:** maximise mean out-of-fold PR-AUC across all 5 folds.

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """Optuna objective: maximise cross-validated PR-AUC for CatBoost."""
    params = {
        "depth":              trial.suggest_int("depth", 4, 8),
        "learning_rate":      trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "l2_leaf_reg":        trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "random_strength":    trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "min_data_in_leaf":   trial.suggest_int("min_data_in_leaf", 5, 100),
        "max_ctr_complexity": trial.suggest_int("max_ctr_complexity", 1, 3),
        "one_hot_max_size":   trial.suggest_int("one_hot_max_size", 2, 12),
        "model_size_reg":     trial.suggest_float("model_size_reg", 0.1, 10.0, log=True),
        "balance_strategy":   trial.suggest_categorical("balance_strategy", ["auto", "manual"]),
    }

    if params.pop("balance_strategy") == "auto":
        params["auto_class_weights"] = trial.suggest_categorical(
            "auto_class_weights", ["Balanced", "SqrtBalanced"]
        )
    else:
        params["scale_pos_weight"] = trial.suggest_float("scale_pos_weight", 10.0, 50.0)

    fixed_params = dict(
        iterations=2000,
        loss_function="Logloss",
        eval_metric="PRAUC",
        verbose=0,
        allow_writing_files=False,
        random_seed=RANDOM_STATE,
        thread_count=-1,
    )

    fold_pr_aucs = []
    for fold_idx, (train_pool, val_pool, y_va) in enumerate(precomputed_pools):
        cb = CatBoostClassifier(**fixed_params, **params)
        cb.fit(train_pool, eval_set=val_pool,
               early_stopping_rounds=150, use_best_model=True)

        val_prob     = cb.predict_proba(val_pool)[:, 1]
        fold_pr_auc  = average_precision_score(y_va, val_prob)
        fold_pr_aucs.append(fold_pr_auc)

        trial.report(np.mean(fold_pr_aucs), step=fold_idx)

        # Warmup constraint: do not prune before fold 1 to avoid
        # cutting potentially optimal grids due to early variance.
        if fold_idx > 0 and trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_pr_aucs))

### 8.3 Run / Load Optimisation

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

if RUN_OPTUNA:
    study = optuna.create_study(
        direction="maximize",
        study_name="catboost_credit_default",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=1),
    )
    print("[INFO] Launching Optuna optimisation (50 trials)...")
    study.optimize(objective, n_trials=50, n_jobs=1)
    print(f"[INFO] Best PR-AUC: {study.best_value:.5f}")
else:
    study = None
    print("[INFO] RUN_OPTUNA=False — using pre-cached optimal parameters.")
    print("[INFO] Set RUN_OPTUNA=True to re-run the full 50-trial search.")

In [ ]:
# Display optimisation history if a live study was run
if study is not None:
    best_params = study.best_params
    imbalance_mode = "Auto weighting" if "auto_class_weights" in best_params else "Manual scaling"

    print(f"Best CV PR-AUC  : {study.best_value:.5f}")
    print(f"Imbalance mode  : {imbalance_mode}")
    print("Best parameters :")
    for k, v in best_params.items():
        print(f"  {k:<28}: {v}")

    fig = optuna.visualization.plot_optimization_history(study)
    fig.update_layout(
        title=dict(text="<b>Optuna Optimisation History</b>", font=dict(size=14)),
        width=850, height=450, template="plotly_white",
    )
    fig.show()
else:
    print("[INFO] Visualisation skipped (RUN_OPTUNA=False).")
    print("[INFO] Pipeline will train with DEFAULT_BEST_PARAMS.")

### 8.4 Final Ensemble Training

We resolve the best hyperparameters — from a live Optuna run or from
`DEFAULT_BEST_PARAMS` — and retrain one `CatBoostClassifier` per fold.
Each fold model gets a **unique random seed** (base + fold index) to
introduce controlled diversity into the ensemble.

In [ ]:
# ── Resolve best parameters (live study OR pre-cached fallback) ───────────────
best_params = study.best_params if study is not None else DEFAULT_BEST_PARAMS.copy()

# Normalise: ensure loss_function key is present and balance_strategy removed
best_params.pop("balance_strategy", None)
best_params.setdefault("loss_function", "Logloss")

final_fixed_params = dict(
    iterations=2500,
    eval_metric="PRAUC",
    verbose=200,
    allow_writing_files=False,
    random_seed=RANDOM_STATE,
    thread_count=-1,
)
final_cb_params = {**final_fixed_params, **best_params}

# ── Ensemble cross-validation loop ───────────────────────────────────────────
final_models = []
oof_preds    = np.zeros(len(X_all))

print("Training final CatBoost ensemble (StratifiedGroupKFold)...")

for fold_idx, (train_idx, val_idx) in enumerate(splits):
    train_pool, val_pool, y_va = precomputed_pools[fold_idx]

    fold_params = final_cb_params.copy()
    fold_params["random_seed"] = RANDOM_STATE + fold_idx  # ensemble diversity

    model = CatBoostClassifier(**fold_params)
    model.fit(train_pool, eval_set=val_pool,
              early_stopping_rounds=200, use_best_model=True)

    val_prob = model.predict_proba(val_pool)[:, 1]
    oof_preds[val_idx] = val_prob
    final_models.append(model)

    fold_auc = average_precision_score(y_va, val_prob)
    print(f"  Fold {fold_idx} complete — PR-AUC: {fold_auc:.4f}")

### 8.5 Test Set Inference

In [ ]:
# ── Replace None with pd.read_csv(...) if a test file is available ────────────
X_test = None  # e.g. pd.read_csv("test_df.csv", sep="\t")[FINAL_FEATURES]

if X_test is not None:
    X_test_cb, _ = prepare_for_trees(X_test, X_test, final_cat)
    X_test_cb.columns = sanitize_names(X_test_cb.columns.tolist())
    test_pool  = Pool(X_test_cb, cat_features=cat_feature_indices)

    # Soft-voting ensemble average
    test_preds = np.mean(
        [m.predict_proba(test_pool)[:, 1] for m in final_models], axis=0
    )
    print(f"Test inference complete. Predictions shape: {test_preds.shape}")
else:
    print("[INFO] X_test is None — skipping test inference.")
    print("[INFO] Set X_test = pd.read_csv(...) to generate predictions.")

## 9. Results & Evaluation

### 9.1 Model Comparison: Logistic Regression vs CatBoost Ensemble

In [ ]:
lr_mean = lr_metrics_df.mean(numeric_only=True)

oof_pr_auc  = average_precision_score(y_all, oof_preds)
oof_roc_auc = roc_auc_score(y_all, oof_preds)
oof_best_t, oof_best_f1 = find_optimal_f1(y_all, oof_preds)
oof_y_pred  = (oof_preds >= oof_best_t).astype(int)

comparison = pd.DataFrame({
    "Logistic Regression (mean OOF)": {
        "PR-AUC":    round(lr_mean["pr_auc"],  5),
        "ROC-AUC":   round(lr_mean["roc_auc"], 5),
        "F1":        round(lr_mean["f1"],       5),
        "Precision": round(lr_mean["precision"],5),
        "Recall":    round(lr_mean["recall"],   5),
    },
    "CatBoost Ensemble (OOF)": {
        "PR-AUC":    round(oof_pr_auc,  5),
        "ROC-AUC":   round(oof_roc_auc, 5),
        "F1":        round(oof_best_f1, 5),
        "Precision": round(precision_score(y_all, oof_y_pred, zero_division=0), 5),
        "Recall":    round(recall_score(y_all, oof_y_pred,    zero_division=0), 5),
    },
}).T

display(comparison.style.highlight_max(axis=0, color="#c8f7c5").format("{:.5f}"))

### 9.2 Validation Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.set_theme(style="white")

# ── PR Curve ──────────────────────────────────────────────────────────────────
precision_curve, recall_curve, _ = precision_recall_curve(y_all, oof_preds)
axes[0].plot(recall_curve, precision_curve, color="#e66101", lw=2.5,
             label=f"CatBoost OOF ensemble (AUC={oof_pr_auc:.4f})")
axes[0].axhline(y=np.mean(y_all), color="dimgray", linestyle="--", alpha=0.7,
                label=f"Baseline rate ({np.mean(y_all):.4%})")
axes[0].set_title("Precision-Recall Curve", fontsize=12, fontweight="bold", pad=12)
axes[0].set_xlabel("Recall (True Positive Rate)")
axes[0].set_ylabel("Precision (Positive Predictive Value)")
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.0])
axes[0].legend(frameon=False, loc="upper right")
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)
axes[0].grid(axis="both", alpha=0.15)

# ── ROC Curve ─────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_all, oof_preds)
axes[1].plot(fpr, tpr, color="#5e3c99", lw=2.5,
             label=f"CatBoost OOF ensemble (AUC={oof_roc_auc:.4f})")
axes[1].plot([0, 1], [0, 1], color="dimgray", linestyle="--", alpha=0.5,
             label="Random guess baseline")
axes[1].set_title("ROC Curve", fontsize=12, fontweight="bold", pad=12)
axes[1].set_xlabel("False Positive Rate (FPR)")
axes[1].set_ylabel("True Positive Rate (TPR / Recall)")
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.0])
axes[1].legend(frameon=False, loc="lower right")
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
axes[1].grid(axis="both", alpha=0.15)

plt.suptitle("CatBoost Ensemble — OOF Validation Curves", fontsize=14,
             weight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 9.3 Predicted Probability Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

df_probs = pd.DataFrame({"Probability": oof_preds, "Target": y_all})
sns.kdeplot(data=df_probs, x="Probability", hue="Target",
            fill=True, common_norm=False, palette="Dark2", alpha=0.35, ax=ax)

ax.set_xlim(0, 1)
ax.set_title("Predicted Probability Distribution by Class", fontsize=12,
             fontweight="bold", pad=12)
ax.set_xlabel("Predicted Probability of Default (Class 1)")
ax.set_ylabel("Kernel Density (KDE)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.15)
plt.tight_layout()
plt.show()

A well-calibrated model produces **clearly separated KDE peaks** —
the Non-Default (0) mass concentrated near 0 and the Default (1) mass
shifted toward 1. Visible separation here confirms the ensemble has
learned a meaningful risk score rather than collapsing to a degenerate
distribution.

### 9.4 Feature Importance (Ensemble Average)

In [ ]:
safe_feature_names = sanitize_names(FINAL_FEATURES)

importance_dfs = [
    pd.Series(m.get_feature_importance(), index=safe_feature_names)
    for m in final_models
]
mean_importance = pd.concat(importance_dfs, axis=1).mean(axis=1)
top20 = mean_importance.sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 7))
top20.plot(kind="barh", ax=ax, color=PALETTE["primary"])
ax.set_title("Top-20 Feature Importances — Mean across Ensemble Folds",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Mean Feature Importance Score")
ax.set_ylabel("Feature Name")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

### 9.5 Business Threshold Analysis

> **The PR-AUC curve answers "how good is the model overall?"**  
> This section answers the operational question: **"If we set a specific business target — what do we get?"**

A bank's risk team rarely operates at the optimal F1 threshold.  
In practice, the decision depends on the **cost trade-off**:

| Scenario | Priority | Threshold direction |
|---|---|---|
| Minimise loan losses (conservative) | High Recall → catch more defaults | Lower threshold |
| Minimise false flags (aggressive growth) | High Precision → flag only certain defaults | Raise threshold |

Below we sweep **Recall targets** and report the exact Precision and threshold at each operating point.

In [ ]:
# ── Threshold sweep across business recall targets ────────────────────────────
from sklearn.metrics import precision_recall_curve, confusion_matrix

precisions_curve, recalls_curve, thresholds_curve = precision_recall_curve(y_all, oof_preds)

recall_targets = [0.60, 0.70, 0.80, 0.90]
rows = []

for target_recall in recall_targets:
    # Find the threshold where recall is closest to the target (from above)
    # We want recall >= target, so we search where the curve first exceeds it
    mask = recalls_curve >= target_recall
    if not mask.any():
        continue

    # Among all points with recall >= target, pick the one with highest precision
    best_idx = precisions_curve[mask].argmax()
    candidates_prec = precisions_curve[mask]
    candidates_rec  = recalls_curve[mask]
    candidates_thr  = thresholds_curve[mask[:-1]]  # thresholds array is 1 shorter

    idx = candidates_prec.argmax()
    achieved_precision = candidates_prec[idx]
    achieved_recall    = candidates_rec[idx]
    threshold          = candidates_thr[idx] if idx < len(candidates_thr) else thresholds_curve[-1]

    y_pred_t = (oof_preds >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_all, y_pred_t).ravel()

    total_defaults   = int(y_all.sum())
    caught_defaults  = int(tp)
    false_alarms     = int(fp)
    missed_defaults  = int(fn)

    rows.append({
        "Recall target":      f"{target_recall:.0%}",
        "Achieved recall":    f"{achieved_recall:.1%}",
        "Precision":          f"{achieved_precision:.1%}",
        "Threshold":          f"{threshold:.3f}",
        "Defaults caught":    f"{caught_defaults:,} / {total_defaults:,}",
        "False alarms":       f"{false_alarms:,}",
        "Missed defaults":    f"{missed_defaults:,}",
    })

threshold_df = pd.DataFrame(rows)
display(threshold_df.style.set_caption("Business Operating Points — Recall vs Precision Trade-off"))

**How to read the table:**

- **Recall target** — the share of all actual defaults the model *catches*
- **Precision** — of the clients the model *flags*, what share truly default
- **False alarms** — non-defaulters incorrectly flagged (cost = manual review / rejected good clients)
- **Missed defaults** — defaulters the model missed (cost = actual credit loss)

> **Example (Recall = 80%):** The model flags X clients.  
> Of those, Y% are genuine defaulters (`Precision`).  
> The bank catches 80% of all defaults while reviewing only a targeted subset of applications.

In [ ]:
# ── Visual: PR curve with operating points marked ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(recalls_curve, precisions_curve,
        color=PALETTE["primary"], lw=2.5, label="CatBoost OOF ensemble")
ax.axhline(y_all.mean(), color="dimgray", linestyle="--", alpha=0.6,
           label=f"Baseline rate ({y_all.mean():.2%})")

# Plot each operating point
colors_op = ["#C73E1D", "#F18F01", "#2E86AB", "#3A7D44"]
for i, row in enumerate(rows):
    r_val = float(row["Achieved recall"].strip("%")) / 100
    p_val = float(row["Precision"].strip("%")) / 100
    ax.scatter(r_val, p_val, s=120, zorder=5,
               color=colors_op[i], label=f"@ Recall={row['Recall target']} → Precision={row['Precision']}")
    ax.annotate(
        f"  R={row['Achieved recall']}\n  P={row['Precision']}",
        xy=(r_val, p_val),
        fontsize=9,
        color=colors_op[i],
        va="center",
    )

ax.set_title("Precision-Recall Curve with Business Operating Points",
             fontsize=13, fontweight="bold", pad=14)
ax.set_xlabel("Recall — share of defaults caught")
ax.set_ylabel("Precision — accuracy among flagged clients")
ax.set_xlim([0.0, 1.05])
ax.set_ylim([0.0, 1.0])
ax.legend(frameon=False, fontsize=9, loc="upper right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="both", alpha=0.15)
plt.tight_layout()
plt.show()

---

## 9.6 Executive Summary & Business Conclusions

### Task
Binary classification of credit default risk (`gb=1`) on severely
imbalanced data (~9% default rate) with a grouped client structure
(one client may appear in multiple rows).

### Results

| Metric | LR Baseline | CatBoost Ensemble | Lift |
|---|---|---|---|
| **PR-AUC** | ~0.030 | **~0.199** | **~6.6×** |
| **ROC-AUC** | ~0.540 | **~0.821** | **~1.52×** |

### Engineering decisions & rationale

| Decision | Rationale |
|---|---|
| `StratifiedGroupKFold` by `id` | Prevents data leakage from client appearing in both train and val |
| PR-AUC as primary metric | Robust to heavy class imbalance; not distorted by true negatives |
| `RobustScaler` for LR | Neutralises the effect of extreme outliers on linear weights |
| CatBoost native CTR encoding | Handles high-cardinality categories without manual binning |
| Pre-computed `Pool` objects | Cuts Optuna iteration time by caching transformations |
| `MedianPruner` + fold-0 warmup | Drops unpromising trials early without cutting too aggressively |
| Ensemble (5 fold models) | Reduces variance via soft-voting; each fold seeded uniquely |

### Next steps (production roadmap)
- **SHAP analysis** — feature attribution for regulatory explainability
- **Probability calibration** (Platt / Isotonic) — trustworthy risk scores for portfolio decisions
- **Business cost matrix** — threshold tuning based on FN vs FP financial impact
- **Distribution shift monitoring** — detect covariate drift in production data streams
- **Incremental retraining** — periodic refit as new loan data arrives